# Problema 1

Dado un hash, encontrar la clave de 8 dígitos que lo genera.

In [1]:
import hashlib

In [2]:
# Prueba con clave conocida

objetivo = "12345678"
hash_objetivo = hashlib.sha256(objetivo.encode()).hexdigest()
print(hash_objetivo)

for i in range(100_000_000): # rango que contiene desde (00000000, hasta 99999999)
    cadena = str(i).zfill(8) # se rellena con 0s iniciales para los números con menos de 8 dígitos
    hash = hashlib.sha256(cadena.encode()).hexdigest()

    if hash == hash_objetivo: # se compara el hash de la clave candidata con el objetivo
        print(f"Clave encontrada: {cadena}")
        break

ef797c8118f02dfb649607dd5d3f8c7623048c9c063d532cc95c5ed7a898a64f
Clave encontrada: 12345678


In [4]:
hash_objetivo = input("Ingrese el hash objetivo: ")

for i in range(100_000_000): # rango que contiene desde (00000000, hasta 99999999)
    cadena = str(i).zfill(8) # se rellena con 0s iniciales para los números con menos de 8 dígitos
    hash = hashlib.sha256(cadena.encode()).hexdigest()

    if hash == hash_objetivo: # se compara el hash de la clave candidata con el objetivo
        print(f"Clave encontrada: {cadena}")
        break

Clave encontrada: 99999999


# Problema 2 

Construcción de Árbol Merkel con base a un conjunto de transacciones

In [7]:
import json

def sha256(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()

class MerkelTree:
    def __init__(self, transacciones: list):
        """
        Constructor que inicializa el árbol como una matriz bidimensional inmutable
        que funciona bajo el concepto de pila.
        Detecta automáticamente si los datos son texto plano o diccionarios complejos.
        """
        if not transacciones:
            self.matriz = ((),)
            self.transacciones = ()
            return
        
        self.transacciones = tuple(transacciones)
        
        hojas = []
        for tx in transacciones:
            # Si es un diccionario/objeto complejo
            if isinstance(tx, dict):
                serializacion = json.dumps(tx, sort_keys=True)
                hojas.append(sha256(serializacion))
            # Si ya es un texto plano (string)
            else:
                hojas.append(sha256(str(tx)))
            
        self.matriz = [tuple(hojas)]
        
        # Construimos los siguientes niveles hacia arriba
        while len(self.matriz[-1]) > 1:
            capa_anterior = list(self.matriz[-1])
            
            # Si el nivel es impar, duplicamos el último elemento
            if len(capa_anterior) % 2 != 0:
                capa_anterior.append(capa_anterior[-1])
               
            siguiente_capa = []
            for i in range(0, len(capa_anterior), 2):
                combinacion = capa_anterior[i] + capa_anterior[i + 1]
                siguiente_capa.append(sha256(combinacion))
            
            self.matriz.append(tuple(siguiente_capa))
        
        self.matriz = tuple(self.matriz)
    
    def get_root(self) -> str:
        """Devuelve el hash raíz del árbol (el tope de la pila)."""
        return self.matriz[-1][0] if self.matriz else ""
    
    def get_proof(self, indice: int) -> list:
        """Genera los pasos de la prueba (Merkle Proof) para una transacción."""
        if indice < 0 or indice >= len(self.matriz):
            return []
        
        prueba = []
        indice_actual = indice
        
        for capa in self.matriz[:-1]:
            es_impar = indice_actual % 2 == 1
            indice_pareja = indice_actual - 1 if es_impar else min(indice_actual + 1, len(capa) - 1)
            
            prueba.append({
                "posicion": "left" if es_impar else "right",
                "hash": capa[indice_pareja]
            })
            indice_actual //= 2
            
        return prueba
    
    def generar_reporte_texto(self) -> str:
        """Genera una cadena de texto con la estructura jerárquica del árbol."""
        root = self.get_root()
        hojas = self.matriz[0]
        lineas = []
        lineas.append(f"MERKLE ROOT FINAL:\n{root}\n")
        lineas.append("[RAÍZ]")
        lineas.append(f" └── {root[:8]}... (HASH: {root})")
        
        if len(self.matriz) > 2:
            lineas.append("\n[NODOS INTERMEDIOS]")
            for i in range(len(self.matriz) - 2, 0, -1):
                for idx, h in enumerate(self.matriz[i]):
                    lineas.append(f" ├── H{i}{idx+1}: {h[:8]}...")
        
        lineas.append("\n[HOJAS Y TRANSACCIONES]")
        for i, (tx, h) in enumerate(zip(self.transacciones, hojas), 1):
            lineas.append(f" ├── T{i} ({tx})")
            lineas.append(f" │    └── Hash H{i}: {h[:8]}... (HASH: {h})")
            
        return "\n".join(lineas)
    
    def dibujar_y_guardar_arbol(self, nombre_archivo="resultado_merkle.txt"):
        """Muestra el árbol en consola y exporta el diseño visual a un archivo .txt."""
        reporte = self.generar_reporte_texto()
        
        # Imprimir en consola
        print(reporte)
        
        # Guardar en archivo de texto plano
        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write(reporte)
        print(f"\n[SISTEMA] El reporte visual ha sido guardado con éxito en: '{nombre_archivo}'")
    

In [11]:
banco_datos = [
    "Ana paga 150",  # T1
    "Luis paga 230",  # T2
    "Carlos paga 80", # T3
    "Maria paga 95"   # T4
]
arbol = MerkelTree(banco_datos)
arbol.dibujar_y_guardar_arbol("mi_arbol.txt")

MERKLE ROOT FINAL:
3dee224f83ea08bc55ac2680bb5eced286ab775fca484acc3eafd757bbb71d93

[RAÍZ]
 └── 3dee224f... (HASH: 3dee224f83ea08bc55ac2680bb5eced286ab775fca484acc3eafd757bbb71d93)

[NODOS INTERMEDIOS]
 ├── H11: 135b71cf...
 ├── H12: f350e823...

[HOJAS Y TRANSACCIONES]
 ├── T1 (Ana paga 150)
 │    └── Hash H1: 7a18bfdc... (HASH: 7a18bfdcfe365d9598cb789513f9239a6cef11c4d2c2a1a6f32677e64777f97c)
 ├── T2 (Luis paga 230)
 │    └── Hash H2: 2f4c9726... (HASH: 2f4c9726fa7bcc9b054a82470b582e2b504b8bd5aae1a4789eaefc76d5a774e1)
 ├── T3 (Carlos paga 80)
 │    └── Hash H3: 19b08e51... (HASH: 19b08e517d817544fc0baec4aa06a8c0ba9452be3b3e00c03af62c7c6da9fc46)
 ├── T4 (Maria paga 95)
 │    └── Hash H4: c537d0cc... (HASH: c537d0cc84287857e88596f027c7ef63a2b0964c358336495961aef92df264cd)

[SISTEMA] El reporte visual ha sido guardado con éxito en: 'mi_arbol.txt'


In [8]:
class MerkleProof:
    @staticmethod
    def verificar(prueba: list, hash_objetivo: str, raiz: str) -> bool:
        """Verifica si una prueba reconstruye con éxito la raíz provista."""
        hash_actual = hash_objetivo
        for paso in prueba:
            if paso["posicion"] == "left":
                combinado = paso["hash"] + hash_actual
            else:
                combinado = hash_actual + paso["hash"]
            hash_actual = sha256(combinado)
        return hash_actual == raiz